# Time-Series Patterns

## Quick notes
### What this covers
- Calendar (date spine) tables built with a recursive CTE, so days with no data show up as zero.
- Running totals, moving averages, month-over-month and year-over-year change, period-to-date numbers.
- Weekly and monthly rollups with `strftime`, cumulative distinct counts, simple trend forecasting, anomaly flags, rate of change, forward fill.
### Key points
- Real data has holes: no sales on a Sunday means no row, not a row with 0. Build the calendar first, then `LEFT JOIN` the data onto it.
- Window frames count rows (`ROWS BETWEEN`), not days. They are only "7 days" if every day has exactly one row. The date spine guarantees that.
- `LAG(x, n)` looks n rows back. It is only "last month" or "last year" when no month is missing.
- SQLite has no `DATE_TRUNC` or `DATEDIFF`. Use `strftime`, `date(x, 'start of month')` and `julianday(a) - julianday(b)`.
- Store dates as ISO text (`YYYY-MM-DD`). They then sort and compare correctly as plain text.
### Gotchas (the part worth revising twice)
- A `GROUP BY day` only returns days that exist. Missing days silently vanish from charts and averages.
- Filtering with `WHERE` runs before the window function. If you want a window over all history but only some rows shown, compute the window first (CTE or subquery) and filter outside.
- A moving average over the first rows uses a short window. Decide whether to show it or hide it (`NULL` until the window is full).
- Impossible dates do not raise errors. `date('2025-13-01')` quietly returns `NULL`, and `date('2025-02-30')` is shifted to `2025-03-02`. Bad dates either disappear or move.
- Percent change divides by the old value: guard against zero with `NULLIF(old, 0)`.
- Weeks have no single definition. `%W` starts on Monday, `%U` on Sunday, and week 0 exists. Compute the week start date instead.
- A regression or z-score in SQL is a quick sanity tool, not a statistics package.
### When it matters
- Revenue and traffic dashboards, KPI reports, capacity planning, monitoring ("is today unusual?").
- Anywhere a chart should show a flat zero instead of a missing point.
### One-line recap example
- `WITH RECURSIVE d(day) AS (SELECT '2025-01-01' UNION ALL SELECT date(day,'+1 day') FROM d WHERE day < '2025-01-31') SELECT day, coalesce(sum(amount),0) FROM d LEFT JOIN sales ON sale_date = day GROUP BY day;` means "every day of January, with 0 when nothing sold".

## 01. Seed data
- `sales`: individual sales with gaps (no sales on some days). `monthly_revenue`: 18 months, Jan 2024 to Jun 2025.
- `visits`: which user visited on which day. `api_latency`: one reading per day. `price_changes`: a price that only changes occasionally. `meter`: irregular readings of a cumulative meter.
- All dates are fixed, nothing depends on today's date.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE sales (sale_id INTEGER PRIMARY KEY, sale_date TEXT NOT NULL, amount INTEGER NOT NULL);
INSERT INTO sales VALUES
    (1,'2025-01-01',120),(2,'2025-01-01',80),(3,'2025-01-02',150),(4,'2025-01-04',90),
    (5,'2025-01-05',200),(6,'2025-01-07',60),(7,'2025-01-08',110),(8,'2025-01-08',40),
    (9,'2025-01-11',300),(10,'2025-01-12',130),(11,'2025-01-14',70),
    (12,'2025-02-01',90),(13,'2025-02-03',110),(14,'2025-02-06',60);
CREATE TABLE monthly_revenue (month TEXT PRIMARY KEY, revenue INTEGER NOT NULL);
INSERT INTO monthly_revenue VALUES
    ('2024-01',100),('2024-02',104),('2024-03',110),('2024-04',108),('2024-05',115),('2024-06',120),
    ('2024-07',118),('2024-08',125),('2024-09',130),('2024-10',128),('2024-11',140),('2024-12',160),
    ('2025-01',112),('2025-02',115),('2025-03',123),('2025-04',121),('2025-05',130),('2025-06',136);
CREATE TABLE visits (visit_date TEXT NOT NULL, user_id INTEGER NOT NULL);
INSERT INTO visits VALUES
    ('2025-01-01',1),('2025-01-01',2),('2025-01-02',1),('2025-01-02',3),('2025-01-02',4),
    ('2025-01-03',2),('2025-01-03',3),('2025-01-04',5),('2025-01-05',1),('2025-01-05',5),('2025-01-05',6);
CREATE TABLE api_latency (day TEXT PRIMARY KEY, ms INTEGER NOT NULL);
INSERT INTO api_latency VALUES
    ('2025-01-01',98),('2025-01-02',102),('2025-01-03',101),('2025-01-04',99),('2025-01-05',103),
    ('2025-01-06',100),('2025-01-07',97),('2025-01-08',104),('2025-01-09',100),('2025-01-10',412),
    ('2025-01-11',101),('2025-01-12',99),('2025-01-13',102),('2025-01-14',98);
CREATE TABLE price_changes (change_date TEXT PRIMARY KEY, price REAL NOT NULL);
INSERT INTO price_changes VALUES ('2025-01-01',10.0),('2025-01-04',12.5),('2025-01-09',11.0);
CREATE TABLE meter (read_date TEXT PRIMARY KEY, kwh INTEGER NOT NULL);
INSERT INTO meter VALUES ('2025-01-01',1000),('2025-01-04',1090),('2025-01-05',1120),('2025-01-10',1300);
SELECT * FROM sales ORDER BY sale_id;

sale_id | sale_date  | amount
--------+------------+-------
1       | 2025-01-01 | 120   
2       | 2025-01-01 | 80    
3       | 2025-01-02 | 150   
4       | 2025-01-04 | 90    
5       | 2025-01-05 | 200   
6       | 2025-01-07 | 60    
7       | 2025-01-08 | 110   
8       | 2025-01-08 | 40    
9       | 2025-01-11 | 300   
10      | 2025-01-12 | 130   
11      | 2025-01-14 | 70    
12      | 2025-02-01 | 90    
13      | 2025-02-03 | 110   
14      | 2025-02-06 | 60    
(14 rows)



## 02. Date tools in SQLite
- `date(x, modifiers)` shifts and truncates: `'+1 day'`, `'-1 year'`, `'start of month'`, `'weekday 1'` (next Monday, or the same day if it is Monday).
- `strftime('%Y-%m', x)` builds month labels. `%w` is the weekday (0 = Sunday), `%j` the day of year.
- `julianday(a) - julianday(b)` is the number of days between two dates.
- Last day of a month: go to the start of next month, then back one day.

In [3]:
%%sql
SELECT date('2025-01-31', '+1 month') AS plus_month,
       date('2025-03-15', 'start of month') AS month_start,
       date('2025-03-15', 'start of month', '+1 month', '-1 day') AS month_end,
       strftime('%Y-%m', '2025-03-15') AS month_label,
       strftime('%w', '2025-01-01') AS weekday_0_is_sunday,
       julianday('2025-03-01') - julianday('2025-02-01') AS days_between;

plus_month | month_start | month_end  | month_label | weekday_0_is_sunday | days_between
-----------+-------------+------------+-------------+---------------------+-------------
2025-03-03 | 2025-03-01  | 2025-03-31 | 2025-03     | 3                   | 28.0        
(1 row)



- Note `'2025-01-31' +1 month` gives `2025-03-03`: SQLite normalises "Feb 31" forward. For month arithmetic, start from `'start of month'` to avoid surprises.

## 03. A date spine with a recursive CTE
- A recursive CTE starts with one row (the anchor) and repeatedly adds the next day until the stop condition fails.
- The result is a table of every day in the range. It is the base for every "fill the gaps" report.
- Always include the `WHERE day < end` stop, otherwise the recursion never ends.
- Recursive CTEs are explained in more depth in 04-Patterns/06-Hierarchies-And-Graphs.

In [4]:
%%sql
WITH RECURSIVE spine(day) AS (
    SELECT '2025-01-01'
    UNION ALL
    SELECT date(day, '+1 day') FROM spine WHERE day < '2025-01-07'
)
SELECT day, strftime('%w', day) AS weekday FROM spine ORDER BY day;

day        | weekday
-----------+--------
2025-01-01 | 3      
2025-01-02 | 4      
2025-01-03 | 5      
2025-01-04 | 6      
2025-01-05 | 0      
2025-01-06 | 1      
2025-01-07 | 2      
(7 rows)



## 04. Fill missing days with zero
- `GROUP BY sale_date` only returns days that have sales. Days 3, 6, 9, 10 and 13 of January are missing.
- `LEFT JOIN` the sales onto the spine, group by the spine day, and wrap the sum in `COALESCE(..., 0)`.
- Put the spine on the left. If the data is on the left you will not get the empty days back.

In [5]:
%%sql
SELECT sale_date, sum(amount) AS total
FROM sales WHERE sale_date <= '2025-01-14'
GROUP BY sale_date ORDER BY sale_date;

sale_date  | total
-----------+------
2025-01-01 | 200  
2025-01-02 | 150  
2025-01-04 | 90   
2025-01-05 | 200  
2025-01-07 | 60   
2025-01-08 | 150  
2025-01-11 | 300  
2025-01-12 | 130  
2025-01-14 | 70   
(9 rows)



In [6]:
%%sql
CREATE VIEW daily AS
WITH RECURSIVE spine(day) AS (
    SELECT '2025-01-01'
    UNION ALL
    SELECT date(day, '+1 day') FROM spine WHERE day < '2025-02-07'
)
SELECT s.day, COALESCE(SUM(f.amount), 0) AS total, COUNT(f.sale_id) AS n_sales
FROM spine s LEFT JOIN sales f ON f.sale_date = s.day
GROUP BY s.day;
SELECT * FROM daily WHERE day <= '2025-01-14' ORDER BY day;

day        | total | n_sales
-----------+-------+--------
2025-01-01 | 200   | 2      
2025-01-02 | 150   | 1      
2025-01-03 | 0     | 0      
2025-01-04 | 90    | 1      
2025-01-05 | 200   | 1      
2025-01-06 | 0     | 0      
2025-01-07 | 60    | 1      
2025-01-08 | 150   | 2      
2025-01-09 | 0     | 0      
2025-01-10 | 0     | 0      
2025-01-11 | 300   | 1      
2025-01-12 | 130   | 1      
2025-01-13 | 0     | 0      
2025-01-14 | 70    | 1      
(14 rows)



## 05. Running totals
- `SUM(x) OVER (ORDER BY day)` gives the cumulative total so far.
- With `ORDER BY` and no frame, the default frame is `RANGE ... UNBOUNDED PRECEDING AND CURRENT ROW`. Ties get the same total. Writing `ROWS UNBOUNDED PRECEDING` explicitly is safer.
- More on frames: 03-Window-Functions/01-Window-Functions.

In [7]:
%%sql
SELECT day, total,
       SUM(total) OVER (ORDER BY day ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total
FROM daily WHERE day <= '2025-01-14' ORDER BY day;

day        | total | running_total
-----------+-------+--------------
2025-01-01 | 200   | 200          
2025-01-02 | 150   | 350          
2025-01-03 | 0     | 350          
2025-01-04 | 90    | 440          
2025-01-05 | 200   | 640          
2025-01-06 | 0     | 640          
2025-01-07 | 60    | 700          
2025-01-08 | 150   | 850          
2025-01-09 | 0     | 850          
2025-01-10 | 0     | 850          
2025-01-11 | 300   | 1150         
2025-01-12 | 130   | 1280         
2025-01-13 | 0     | 1280         
2025-01-14 | 70    | 1350         
(14 rows)



## 06. Moving averages with ROWS BETWEEN
- `AVG(total) OVER (ORDER BY day ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` is a 3-day moving average.
- It is a "3-day" average only because the spine has exactly one row per day. On raw data with gaps the same frame would span more days.
- The first two rows use a shorter window. The version below hides them with `CASE` on `COUNT(*) OVER (...)`.

In [8]:
%%sql
SELECT day, total,
       round(AVG(total) OVER w, 1) AS ma3_partial,
       CASE WHEN COUNT(*) OVER w = 3 THEN round(AVG(total) OVER w, 1) END AS ma3_full_only
FROM daily WHERE day <= '2025-01-10'
WINDOW w AS (ORDER BY day ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
ORDER BY day;

day        | total | ma3_partial | ma3_full_only
-----------+-------+-------------+--------------
2025-01-01 | 200   | 200.0       | NULL         
2025-01-02 | 150   | 175.0       | NULL         
2025-01-03 | 0     | 116.7       | 116.7        
2025-01-04 | 90    | 80.0        | 80.0         
2025-01-05 | 200   | 96.7        | 96.7         
2025-01-06 | 0     | 96.7        | 96.7         
2025-01-07 | 60    | 86.7        | 86.7         
2025-01-08 | 150   | 70.0        | 70.0         
2025-01-09 | 0     | 70.0        | 70.0         
2025-01-10 | 0     | 50.0        | 50.0         
(10 rows)



## 07. Month-over-month change with LAG
- `LAG(revenue)` brings the previous row's value onto the current row. The first row has no previous row, so it is `NULL`.
- Absolute change is `revenue - prev`. Percent change is `(revenue - prev) / prev`.
- Multiply by `100.0` (not `100`) to avoid integer division. Guard the divisor with `NULLIF(prev, 0)`.

In [9]:
%%sql
SELECT month, revenue,
       LAG(revenue) OVER (ORDER BY month) AS prev,
       revenue - LAG(revenue) OVER (ORDER BY month) AS change,
       round(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
             / NULLIF(LAG(revenue) OVER (ORDER BY month), 0), 1) AS pct_change
FROM monthly_revenue ORDER BY month;

month   | revenue | prev | change | pct_change
--------+---------+------+--------+-----------
2024-01 | 100     | NULL | NULL   | NULL      
2024-02 | 104     | 100  | 4      | 4.0       
2024-03 | 110     | 104  | 6      | 5.8       
2024-04 | 108     | 110  | -2     | -1.8      
2024-05 | 115     | 108  | 7      | 6.5       
2024-06 | 120     | 115  | 5      | 4.3       
2024-07 | 118     | 120  | -2     | -1.7      
2024-08 | 125     | 118  | 7      | 5.9       
2024-09 | 130     | 125  | 5      | 4.0       
2024-10 | 128     | 130  | -2     | -1.5      
2024-11 | 140     | 128  | 12     | 9.4       
2024-12 | 160     | 140  | 20     | 14.3      
2025-01 | 112     | 160  | -48    | -30.0     
2025-02 | 115     | 112  | 3      | 2.7       
2025-03 | 123     | 115  | 8      | 7.0       
2025-04 | 121     | 123  | -2     | -1.6      
2025-05 | 130     | 121  | 9      | 7.4       
2025-06 | 136     | 130  | 6      | 4.6       
(18 rows)



## 08. Year-over-year, and the missing-month trap
- `LAG(revenue, 12)` is "same month last year", but only if there are 12 rows per year with no holes.
- Safer: join the table to itself on the calendar. The previous year of `'2025-03'` is `strftime('%Y-%m', date('2025-03-01', '-1 year'))`.
- Below, April 2024 is deleted from a copy. `LAG(.., 12)` then points at the wrong month, and the join correctly returns `NULL`.

In [10]:
%%sql
SELECT c.month, c.revenue, p.revenue AS same_month_last_year,
       round(100.0 * (c.revenue - p.revenue) / p.revenue, 1) AS yoy_pct
FROM monthly_revenue c
LEFT JOIN monthly_revenue p
       ON p.month = strftime('%Y-%m', date(c.month || '-01', '-1 year'))
WHERE c.month >= '2024-12'
ORDER BY c.month;

month   | revenue | same_month_last_year | yoy_pct
--------+---------+----------------------+--------
2024-12 | 160     | NULL                 | NULL   
2025-01 | 112     | 100                  | 12.0   
2025-02 | 115     | 104                  | 10.6   
2025-03 | 123     | 110                  | 11.8   
2025-04 | 121     | 108                  | 12.0   
2025-05 | 130     | 115                  | 13.0   
2025-06 | 136     | 120                  | 13.3   
(7 rows)



In [11]:
%%sql
CREATE TEMP TABLE gappy AS SELECT * FROM monthly_revenue WHERE month <> '2024-04';
SELECT month, revenue, lag12 AS lag12_wrong, join_right
FROM (
    SELECT g.month, g.revenue,
           LAG(g.revenue, 12) OVER (ORDER BY g.month) AS lag12,
           p.revenue AS join_right
    FROM gappy g
    LEFT JOIN gappy p ON p.month = strftime('%Y-%m', date(g.month || '-01', '-1 year'))
)
WHERE month IN ('2025-03','2025-04','2025-05')
ORDER BY month;

month   | revenue | lag12_wrong | join_right
--------+---------+-------------+-----------
2025-03 | 123     | 104         | 110       
2025-04 | 121     | 110         | NULL      
2025-05 | 130     | 115         | 115       
(3 rows)



- With the hole in April 2024, `LAG(.., 12)` is one month off for the months whose year-ago window crosses the hole (`2025-01` to `2025-04`): `2025-03` is compared with Feb 2024 (104, should be 110), and `2025-04` is compared with Mar 2024 (110, should be `NULL` because April 2024 is missing). From `2025-05` on it lines up again, because the missing row is no longer between the two months. The join is right every time. (The window runs in a subquery so the `WHERE` does not remove history rows.)

## 09. Weekly and monthly rollups with strftime
- Monthly: group by `strftime('%Y-%m', day)`. Weekly: group by the Monday of that week.
- Monday of the week: `date(day, '-6 days', 'weekday 1')`. Step back six days, then move forward to the next Monday (or stay).
- Rolling up the spine (not the raw table) keeps empty weeks as zero.

In [12]:
%%sql
SELECT date(day, '-6 days', 'weekday 1') AS week_start,
       MIN(day) AS first_day, MAX(day) AS last_day,
       SUM(total) AS week_total, SUM(n_sales) AS n_sales
FROM daily
WHERE day <= '2025-01-14'
GROUP BY week_start ORDER BY week_start;

week_start | first_day  | last_day   | week_total | n_sales
-----------+------------+------------+------------+--------
2024-12-30 | 2025-01-01 | 2025-01-05 | 640        | 5      
2025-01-06 | 2025-01-06 | 2025-01-12 | 640        | 5      
2025-01-13 | 2025-01-13 | 2025-01-14 | 70         | 1      
(3 rows)



In [13]:
%%sql
SELECT strftime('%Y-%m', day) AS month, SUM(total) AS month_total, COUNT(*) AS days_in_spine
FROM daily GROUP BY month ORDER BY month;

month   | month_total | days_in_spine
--------+-------------+--------------
2025-01 | 1350        | 31           
2025-02 | 260         | 7            
(2 rows)



- `2025-01-01` is a Wednesday, so the first week starts on Monday `2024-12-30` and only holds 5 days. February is cut off at the 7th because that is where the spine ends (7 days).

## 10. Period-to-date: MTD and YTD
- Month-to-date: `SUM(total) OVER (PARTITION BY month ORDER BY day)`. The running total restarts at each new month.
- Year-to-date works the same with the year in `PARTITION BY`.
- Gotcha: if you filter with `WHERE` first, the window only sees the filtered rows. Here the filter is applied outside, so the January part is still counted.

In [14]:
%%sql
SELECT * FROM (
    SELECT day, total,
           SUM(total) OVER (PARTITION BY strftime('%Y-%m', day) ORDER BY day
                            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS mtd
    FROM daily
)
WHERE day BETWEEN '2025-01-30' AND '2025-02-04'
ORDER BY day;

day        | total | mtd 
-----------+-------+-----
2025-01-30 | 0     | 1350
2025-01-31 | 0     | 1350
2025-02-01 | 90    | 90  
2025-02-02 | 0     | 90  
2025-02-03 | 110   | 200 
2025-02-04 | 0     | 200 
(6 rows)



In [15]:
%%sql
SELECT month, revenue,
       SUM(revenue) OVER (PARTITION BY substr(month, 1, 4) ORDER BY month
                          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS ytd
FROM monthly_revenue WHERE month >= '2024-11'
ORDER BY month;

month   | revenue | ytd
--------+---------+----
2024-11 | 140     | 140
2024-12 | 160     | 300
2025-01 | 112     | 112
2025-02 | 115     | 227
2025-03 | 123     | 350
2025-04 | 121     | 471
2025-05 | 130     | 601
2025-06 | 136     | 737
(8 rows)



## 11. Cumulative distinct counts (total unique users so far)
- You cannot write `COUNT(DISTINCT user_id) OVER (...)`: SQLite does not allow `DISTINCT` in window functions.
- Instead, find each user's first day, count new users per day, then take a running sum of the new users.
- `dau` (distinct users that day) and cumulative uniques answer different questions.

In [16]:
%%sql expect_error
SELECT visit_date, COUNT(DISTINCT user_id) OVER (ORDER BY visit_date) FROM visits;

ERROR: DISTINCT is not supported for window functions



In [17]:
%%sql
WITH first_seen AS (
    SELECT user_id, MIN(visit_date) AS first_day FROM visits GROUP BY user_id
), new_per_day AS (
    SELECT first_day AS day, COUNT(*) AS new_users FROM first_seen GROUP BY first_day
), dau AS (
    SELECT visit_date AS day, COUNT(DISTINCT user_id) AS dau FROM visits GROUP BY visit_date
)
SELECT d.day, d.dau, COALESCE(n.new_users, 0) AS new_users,
       SUM(COALESCE(n.new_users, 0)) OVER (ORDER BY d.day) AS cumulative_unique_users
FROM dau d LEFT JOIN new_per_day n ON n.day = d.day
ORDER BY d.day;

day        | dau | new_users | cumulative_unique_users
-----------+-----+-----------+------------------------
2025-01-01 | 2   | 2         | 2                      
2025-01-02 | 3   | 2         | 4                      
2025-01-03 | 2   | 0         | 4                      
2025-01-04 | 1   | 1         | 5                      
2025-01-05 | 3   | 1         | 6                      
(5 rows)



## 12. Forecasting-lite: a straight-line trend in SQL
- Least squares for `y = intercept + slope * x` needs only sums: `n`, `SUM(x)`, `SUM(y)`, `SUM(x*y)`, `SUM(x*x)`.
- Use month number `x = 0, 1, 2, ...` (from `ROW_NUMBER`). Slope is `(n*Sxy - Sx*Sy) / (n*Sxx - Sx*Sx)`.
- Then predict future x values. This is a rough guide: it ignores seasonality (the December bump below) and gives no confidence range.
- Real forecasting belongs in Python or R. SQL is good enough for a quick sanity check.

In [18]:
%%sql
WITH t AS (
    SELECT ROW_NUMBER() OVER (ORDER BY month) - 1 AS x, month, revenue AS y FROM monthly_revenue
), s AS (
    SELECT COUNT(*) AS n, SUM(x) AS sx, SUM(y) AS sy, SUM(x*y) AS sxy, SUM(x*x) AS sxx FROM t
), fit AS (
    SELECT (n*sxy - sx*sy) * 1.0 / (n*sxx - sx*sx) AS slope,
           (sy - (n*sxy - sx*sy) * 1.0 / (n*sxx - sx*sx) * sx) / n AS intercept, n
    FROM s
)
SELECT round(slope, 3) AS slope_per_month, round(intercept, 2) AS intercept FROM fit;

slope_per_month | intercept
----------------+----------
1.555           | 108.73   
(1 row)



In [19]:
%%sql
WITH t AS (
    SELECT ROW_NUMBER() OVER (ORDER BY month) - 1 AS x, month, revenue AS y FROM monthly_revenue
), s AS (
    SELECT COUNT(*) AS n, SUM(x) AS sx, SUM(y) AS sy, SUM(x*y) AS sxy, SUM(x*x) AS sxx FROM t
), fit AS (
    SELECT (n*sxy - sx*sy) * 1.0 / (n*sxx - sx*sx) AS slope,
           (sy - (n*sxy - sx*sy) * 1.0 / (n*sxx - sx*sx) * sx) / n AS intercept, n
    FROM s
), future(x) AS (VALUES (18), (19), (20))
SELECT strftime('%Y-%m', date('2024-01-01', '+' || x || ' months')) AS month,
       'forecast' AS kind, round(intercept + slope * x, 1) AS value
FROM future, fit
UNION ALL
SELECT month, 'actual', y FROM t WHERE x >= 16
ORDER BY month;

month   | kind     | value
--------+----------+------
2025-05 | actual   | 130  
2025-06 | actual   | 136  
2025-07 | forecast | 136.7
2025-08 | forecast | 138.3
2025-09 | forecast | 139.8
(5 rows)



## 13. Anomalies with a z-score from window averages
- Compare each day to the average and spread of the previous 7 days (`ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING`, which excludes today).
- Standard deviation needs a square root, and `sqrt` is not guaranteed in every SQLite build. So test the squared z-score: `(x - mean)^2 > 9 * variance` means "more than 3 standard deviations away".
- Variance from window averages: `AVG(x*x) - AVG(x)^2`.
- Require a minimum number of history rows, otherwise the first days look odd.

In [20]:
%%sql
WITH w AS (
    SELECT day, ms,
           AVG(ms) OVER h AS mean_prev,
           AVG(ms * ms) OVER h - AVG(ms) OVER h * AVG(ms) OVER h AS var_prev,
           COUNT(*) OVER h AS n_prev
    FROM api_latency
    WINDOW h AS (ORDER BY day ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING)
)
SELECT day, ms, round(mean_prev, 1) AS mean_prev, round(var_prev, 1) AS var_prev,
       CASE WHEN (ms - mean_prev) * (ms - mean_prev) > 9 * var_prev THEN 'ANOMALY' ELSE 'ok' END AS flag
FROM w WHERE n_prev >= 5
ORDER BY day;

day        | ms  | mean_prev | var_prev | flag   
-----------+-----+-----------+----------+--------
2025-01-06 | 100 | 100.6     | 3.4      | ok     
2025-01-07 | 97  | 100.5     | 2.9      | ok     
2025-01-08 | 104 | 100.0     | 4.0      | ok     
2025-01-09 | 100 | 100.9     | 5.0      | ok     
2025-01-10 | 412 | 100.6     | 4.8      | ANOMALY
2025-01-11 | 101 | 145.0     | 11886.3  | ok     
2025-01-12 | 99  | 145.3     | 11860.5  | ok     
2025-01-13 | 102 | 144.7     | 11910.8  | ok     
2025-01-14 | 98  | 145.0     | 11885.7  | ok     
(9 rows)



- Day 10 (412 ms) is flagged. Look at days 11 to 14: the spike is now inside the 7-day history, which inflates the spread, so a second spike would be harder to detect. A common fix is a median or a longer baseline.

## 14. Rate of change between irregular readings
- A cumulative meter is read on irregular dates. Consumption per day is `(kwh - prev_kwh) / days_between`.
- Use `LAG` for the previous reading and `julianday` for the day gap. Never assume the gap is 1.
- Same idea for growth rates: `(new / old) - 1` over the elapsed time.

In [21]:
%%sql
SELECT read_date, kwh,
       kwh - LAG(kwh) OVER (ORDER BY read_date) AS kwh_used,
       CAST(julianday(read_date) - julianday(LAG(read_date) OVER (ORDER BY read_date)) AS INTEGER) AS days,
       round((kwh - LAG(kwh) OVER (ORDER BY read_date)) * 1.0
             / (julianday(read_date) - julianday(LAG(read_date) OVER (ORDER BY read_date))), 1) AS kwh_per_day
FROM meter ORDER BY read_date;

read_date  | kwh  | kwh_used | days | kwh_per_day
-----------+------+----------+------+------------
2025-01-01 | 1000 | NULL     | NULL | NULL       
2025-01-04 | 1090 | 90       | 3    | 30.0       
2025-01-05 | 1120 | 30       | 1    | 30.0       
2025-01-10 | 1300 | 180      | 5    | 36.0       
(4 rows)



## 15. Forward fill: last known value
- A price is stored only on the day it changes. For a daily report, every day needs the latest price on or before that day.
- Simple version: a correlated subquery that picks the most recent change on or before the day.
- Window version: number the groups with `COUNT(price) OVER (ORDER BY day)` (it only increases on days with a value), then take `MAX(price)` inside each group.
- Back-filling the days before the first value is not possible: they stay `NULL`.

In [22]:
%%sql
WITH RECURSIVE spine(day) AS (
    SELECT '2024-12-31' UNION ALL SELECT date(day, '+1 day') FROM spine WHERE day < '2025-01-11'
)
SELECT day,
       (SELECT price FROM price_changes p WHERE p.change_date <= s.day
        ORDER BY p.change_date DESC LIMIT 1) AS price_subquery
FROM spine s ORDER BY day;

day        | price_subquery
-----------+---------------
2024-12-31 | NULL          
2025-01-01 | 10.0          
2025-01-02 | 10.0          
2025-01-03 | 10.0          
2025-01-04 | 12.5          
2025-01-05 | 12.5          
2025-01-06 | 12.5          
2025-01-07 | 12.5          
2025-01-08 | 12.5          
2025-01-09 | 11.0          
2025-01-10 | 11.0          
2025-01-11 | 11.0          
(12 rows)



In [23]:
%%sql
WITH RECURSIVE spine(day) AS (
    SELECT '2024-12-31' UNION ALL SELECT date(day, '+1 day') FROM spine WHERE day < '2025-01-11'
), joined AS (
    SELECT s.day, p.price, COUNT(p.price) OVER (ORDER BY s.day) AS grp
    FROM spine s LEFT JOIN price_changes p ON p.change_date = s.day
)
SELECT day, price AS raw_price, MAX(price) OVER (PARTITION BY grp) AS filled_price
FROM joined ORDER BY day;

day        | raw_price | filled_price
-----------+-----------+-------------
2024-12-31 | NULL      | NULL        
2025-01-01 | 10.0      | 10.0        
2025-01-02 | NULL      | 10.0        
2025-01-03 | NULL      | 10.0        
2025-01-04 | 12.5      | 12.5        
2025-01-05 | NULL      | 12.5        
2025-01-06 | NULL      | 12.5        
2025-01-07 | NULL      | 12.5        
2025-01-08 | NULL      | 12.5        
2025-01-09 | 11.0      | 11.0        
2025-01-10 | NULL      | 11.0        
2025-01-11 | NULL      | 11.0        
(12 rows)



## 16. Date pitfalls: bad vs good
- Bad: dates stored as `'1/5/2025'` or `'05-01-2025'`. They sort alphabetically, so `'12/1/2025'` is "earlier" than `'2/1/2025'`.
- Good: ISO `YYYY-MM-DD` (and `YYYY-MM-DD HH:MM:SS` for times). Text order is time order.
- Unparseable dates return `NULL` from the date functions, which silently drops rows from comparisons. Check with `WHERE date(col) IS NULL`. Impossible days such as Feb 30 are not caught: they roll over to the next month.
- Timestamps with times: `WHERE ts = '2025-01-05'` misses `'2025-01-05 10:30:00'`. Use `ts >= '2025-01-05' AND ts < '2025-01-06'` (half-open range) or `date(ts) = '2025-01-05'`.

In [24]:
%%sql
CREATE TEMP TABLE bad_dates (d TEXT);
INSERT INTO bad_dates VALUES ('2/1/2025'), ('12/1/2025'), ('2025-02-30'), ('2025-13-01');
SELECT d, date(d) AS parsed, 'text order, not date order' AS note FROM bad_dates ORDER BY d;
SELECT '2025-01-05 10:30:00' = '2025-01-05' AS exact_match,
       date('2025-01-05 10:30:00') = '2025-01-05' AS by_date,
       '2025-01-05 10:30:00' >= '2025-01-05' AND '2025-01-05 10:30:00' < '2025-01-06' AS in_half_open_range;

d          | parsed     | note                      
-----------+------------+---------------------------
12/1/2025  | NULL       | text order, not date order
2/1/2025   | NULL       | text order, not date order
2025-02-30 | 2025-03-02 | text order, not date order
2025-13-01 | NULL       | text order, not date order
(4 rows)

exact_match | by_date | in_half_open_range
------------+---------+-------------------
0           | 1       | 1                 
(1 row)



## 17. Putting it together: a daily KPI table
- One query: spine, zero fill, 3-day moving average, running total, and the change compared with the same weekday last week (7 rows back, safe because of the spine).
- This is the shape most dashboards want. Build it once as a view and let reports select from it.

In [25]:
%%sql
SELECT day, total,
       ROUND(AVG(total) OVER (ORDER BY day ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 1) AS ma3,
       SUM(total) OVER (ORDER BY day ROWS UNBOUNDED PRECEDING) AS running_total,
       total - LAG(total, 7) OVER (ORDER BY day) AS vs_last_week
FROM daily
WHERE day BETWEEN '2025-01-01' AND '2025-01-14'
ORDER BY day;

day        | total | ma3   | running_total | vs_last_week
-----------+-------+-------+---------------+-------------
2025-01-01 | 200   | 200.0 | 200           | NULL        
2025-01-02 | 150   | 175.0 | 350           | NULL        
2025-01-03 | 0     | 116.7 | 350           | NULL        
2025-01-04 | 90    | 80.0  | 440           | NULL        
2025-01-05 | 200   | 96.7  | 640           | NULL        
2025-01-06 | 0     | 96.7  | 640           | NULL        
2025-01-07 | 60    | 86.7  | 700           | NULL        
2025-01-08 | 150   | 70.0  | 850           | -50         
2025-01-09 | 0     | 70.0  | 850           | -150        
2025-01-10 | 0     | 50.0  | 850           | 0           
2025-01-11 | 300   | 100.0 | 1150          | 210         
2025-01-12 | 130   | 143.3 | 1280          | -70         
2025-01-13 | 0     | 143.3 | 1280          | 0           
2025-01-14 | 70    | 66.7  | 1350          | 10          
(14 rows)

